In [1]:
import pandas as pd
import numpy as np
import os
GTE_DIR=os.environ["GTE_DIR"]
from glaciation_time_estimator.auxiliary_func.config_reader import read_config
from glaciation_time_estimator.auxiliary_func.chunking_data import ChunkLoader
from glaciation_time_estimator.auxiliary_func.Helper_fun import float_range

In [2]:
config = read_config(
    os.path.join('/wolke_scratch/dnikolo/Glaciation_time_estimator/configs/config_n2o.yaml'))
analyze_year=True
# years=[2007,2008,2009,2010,2011,2012,2013,2014,2015]
years = [year for year in range(2007, 2016)]
# years = [2009]
glac_threshold=config["glac_threshold"]

In [3]:
dataset = ChunkLoader(years,config=config)

KeyboardInterrupt: 

In [4]:
print(dataset)

Dataset chunk loader
                 Analyzing files: {2009: '/wolke_scratch/dnikolo/Final_results/2009_all.parquet'}
                Currently loaded: 2009


In [4]:
def calculate_temp_occurance_rate(dfs):
    combined_cloud_df , _ , _ = dfs
    return combined_cloud_df.groupby("min_temp")["is_glaciating"].mean()*100

In [5]:
[i for i,b in enumerate(np.arange(0.2,0.51,0.1))]

[0, 1, 2, 3]

In [6]:
glac_threshold_arr = [i for i in float_range(0.2,0.51,0.1)]
year_occurance_dict = {}
temp_occurance_dict = {}
for i,glac_threshold in enumerate(glac_threshold_arr):
    config = read_config(
                        os.path.join(f'/wolke_scratch/dnikolo/Glaciation_time_estimator/configs/glac_configs/n2o_2009_thresh_{int(glac_threshold*10):02}.yaml'))
    dataset = ChunkLoader(years,config=config)
    temp_occurance_dict[glac_threshold] = dataset.execute_analysis(calculate_temp_occurance_rate, cloud_columns=["min_temp"], glac_columns=[])

Loading years: 100%|██████████| 9/9 [02:23<00:00, 15.97s/year]


In [9]:
temp_occurance_dict

{0.2: [min_temp
  -36    32.185060
  -30    19.145667
  -24    11.689384
  -18     7.002138
  -12     2.652834
  -6      0.002552
  Name: is_glaciating, dtype: float64,
  min_temp
  -36    32.031792
  -30    19.595763
  -24    12.191870
  -18     7.334120
  -12     2.601946
  -6      0.002273
  Name: is_glaciating, dtype: float64,
  min_temp
  -36    31.465838
  -30    18.729856
  -24    11.374272
  -18     7.010120
  -12     2.579294
  -6      0.002592
  Name: is_glaciating, dtype: float64,
  min_temp
  -36    33.097123
  -30    20.347392
  -24    12.317967
  -18     7.764142
  -12     2.932416
  -6      0.002670
  Name: is_glaciating, dtype: float64,
  min_temp
  -36    32.807627
  -30    21.051878
  -24    13.040408
  -18     8.154505
  -12     3.194303
  -6      0.002862
  Name: is_glaciating, dtype: float64,
  min_temp
  -36    33.347345
  -30    21.112895
  -24    13.151868
  -18     8.023661
  -12     2.991813
  -6      0.003522
  Name: is_glaciating, dtype: float64,
  min_temp


In [13]:
import pandas as pd

# temp_occurance_dict is your dict-of-lists-of-Series.
# years is a list (or array) such that the i-th element is the year corresponding to the i-th Series in each list.

records = []

for threshold, series_list in temp_occurance_dict.items():
    for i, series in enumerate(series_list):
        year = years[i]
        # Convert Series into a DataFrame with columns ['min_temp', 'is_glaciating']
        df = series.reset_index()
        df.columns = ['min_temp', 'occurance_rate']

        # Tag with the current threshold and year
        df['threshold'] = threshold
        df['year'] = year

        records.append(df)

# Stack all small DataFrames into one big DataFrame
combined_df = pd.concat(records, ignore_index=True)

# Optional: reorder columns if you like
combined_df = combined_df[['threshold', 'year', 'min_temp', 'occurance_rate']]
combined_df.to_csv(os.path.join(GTE_DIR,"Result_analysis/Multiyear_analysis/glac_thresh_occurance_rates.csv"))

In [ ]:
# import pandas as pd

# # Suppose temp_occurance_dict is defined like this:
# # {
# #   0.2: [<Series index=min_temp, values=is_glaciating>],
# #   0.3: [<Series index=min_temp, values=is_glaciating>],
# #   ...
# # }

# # Step 1: unwrap each one‐element list to get to the actual Series
# series_dict = {
#     glac_thresh: series_list[0]
#     for glac_thresh, series_list in temp_occurance_dict.items()
# }

# # Step 2: build the DataFrame from the dict of Series
# df = pd.DataFrame(series_dict)

# # Step 3: label the axes
# df.index.name = 'min_temp'
# df.columns.name = 'glac_threshold'

# # Step (2): stack it so that you get a Series whose index is (min_temp, glac_threshold):
# ser = df.stack()
# # Now ser is a pd.Series with a MultiIndex: (min_temp, glac_threshold), and values = is_glaciating.

# # Step (3): turn that into a DataFrame and reset_index:
# df_long = ser.reset_index()
# df_long.columns = ['min_temp', 'glac_threshold', 'occurance_rate']

# # Step (4): add a “year” column (here 2009):
# df_long['year'] = 2009


# df_long.to_csv(os.path.join(GTE_DIR,"Result_analysis/Multiyear_analysis/glac_thresh_occurance_rates.csv"))
# print(df_long)


    min_temp  glac_threshold  occurance_rate  year
0        -36             0.2       31.465838  2009
1        -36             0.3       17.785346  2009
2        -36             0.4       10.121308  2009
3        -36             0.5        5.863809  2009
4        -30             0.2       18.729856  2009
5        -30             0.3       10.318713  2009
6        -30             0.4        5.849524  2009
7        -30             0.5        3.426135  2009
8        -24             0.2       11.374272  2009
9        -24             0.3        6.146662  2009
10       -24             0.4        3.450529  2009
11       -24             0.5        2.034238  2009
12       -18             0.2        7.010120  2009
13       -18             0.3        3.745883  2009
14       -18             0.4        2.134780  2009
15       -18             0.5        1.289744  2009
16       -12             0.2        2.579294  2009
17       -12             0.3        1.261177  2009
18       -12             0.4   